# CatMap pipeline: CSV to GeoJSON

This notebook is the data link of the course pipeline: **CSV -> Python -> GeoJSON -> Leaflet**.

```
geog464-starter/
  data/cat_photos.csv          input table, written by photos_to_csv.ipynb
  data/cat_photos.geojson      output, written by this notebook
  notebooks/photos_to_csv.ipynb
  notebooks/mvp_pipeline.ipynb
  site/index.html, style.css, app.js
  site/photos/                 web copies of the photos
```

Run `photos_to_csv.ipynb` first, then run each cell here in order, top to bottom. Cell 3 writes `data/cat_photos.geojson`, which the web page in `site/` draws.

## 1) Import libraries and load the CSV
We load `../data/cat_photos.csv`, the photo database made by `photos_to_csv.ipynb`.

In [9]:
import pandas as pd
csv_path = "../data/cat_photos.csv"
df = pd.read_csv(csv_path, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
df.head()

,photo,date,lat,lon,name,category,description
0,IMG_8621.HEIC,2024-05-15 14:24,45.520292,-73.580367,NaN,Cat,NaN
1,IMG_8914.HEIC,2024-05-25 13:45,45.522014,-73.581261,NaN,Cat,NaN
2,IMG_1565.HEIC,2024-06-09 10:50,45.521950,-73.581153,NaN,Cat,NaN
3,IMG_0173.HEIC,2024-06-30 19:37,45.520656,-73.581300,NaN,Cat,NaN
4,FullSizeRender.HEIC,2024-06-30 20:03,45.521992,-73.581506,NaN,Cat,NaN


## 2) Check which photos have no location
Photos with blank `lat` or `lon` can't be placed on the map, so the export below skips them. To include one, type its coordinates into `cat_photos.csv` and re-run.

In [10]:
no_location = df[df["lat"].isna() | df["lon"].isna()]
print(f"{len(no_location)} of {len(df)} photos have no location and will be skipped")
no_location

2 of 75 photos have no location and will be skipped


,photo,date,lat,lon,name,category,description
5,B4147EA6-4C48-46D4-A753-67CA3EBC142A.JPG,2024-07-15 20:09,NaN,NaN,NaN,Cat,NaN
15,B91406A3-A1C6-45C4-882C-D54DE2998569.JPG,2024-09-29 21:34,NaN,NaN,NaN,Cat,NaN


## 3) Convert DataFrame to GeoJSON
We iterate through rows and build a GeoJSON `FeatureCollection`. Each feature carries the photo's date, coordinates and the path to its web copy in `site/photos/`, which the map shows in the popup. **Output:** `../data/cat_photos.geojson`

In [11]:
import json
import math
from pathlib import Path

def clean(value):
    # Blank cells come in as NaN; turn them into empty text
    return "" if pd.isna(value) else str(value).strip()

features = []
for _, row in df.iterrows():
    try:
        lat = float(row["lat"])
        lon = float(row["lon"])
    except (TypeError, ValueError):
        continue  # skip bad rows
    if math.isnan(lat) or math.isnan(lon):
        continue  # skip photos without a location

    photo = clean(row.get("photo"))
    feature = {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [lon, lat]},
        "properties": {
            "photo": photo,
            "image": f"photos/{Path(photo).stem}.jpg",  # path as seen from site/index.html
            "date": clean(row.get("date")),
            "lat": round(lat, 6),
            "lon": round(lon, 6),
            "name": clean(row.get("name")),
            "category": clean(row.get("category")),
            "description": clean(row.get("description")),
        },
    }
    features.append(feature)

geojson = {"type": "FeatureCollection", "features": features}
out_path = "../data/cat_photos.geojson"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)
print(f"GeoJSON created with {len(features)} features, written to {out_path}")

GeoJSON created with 73 features, written to ../data/cat_photos.geojson


## 4) See if all checks passed 
- checks to see if the feature count is as expected 
- checks to see if coordiante system is accurate 
- Makes sure that all the photos fit in montreal bounds 
- Checks to see if all the columns (photo,image,date, lat and lon) all exist. 
- Checks if the geoson is 5mb or smaller
- and geometry type 

In [14]:
import os
import geopandas as gpd

# ---- settings: change these for your own data ----
FILE = "../data/cat_photos.geojson"
WEST, EAST, SOUTH, NORTH = -74.0, -73.4, 45.35, 45.75   # a box around Montréal, in degrees
EXPECTED = 73             # 75 photos minus the 2 with no location
POPUP_FIELDS = ["photo", "image", "date", "lat", "lon"]
# ----------------------------------------------------

back = gpd.read_file(FILE)
problems = []
if len(back) != EXPECTED:
    problems.append(f"{len(back)} features, but you expected {EXPECTED}.")
if back.crs is None or back.crs.to_epsg() != 4326:
    problems.append(f"The file is in {back.crs}, not longitude and latitude (EPSG:4326).")
minx, miny, maxx, maxy = back.total_bounds
if not (WEST <= minx and maxx <= EAST and SOUTH <= miny and maxy <= NORTH):
    problems.append(f"Some cats are outside Montréal: x {minx:.3f} to {maxx:.3f}, y {miny:.3f} to {maxy:.3f}.")
missing = [f for f in POPUP_FIELDS if f not in back.columns]
if missing:
    problems.append(f"Missing popup fields: {missing}.")
mb = os.path.getsize(FILE) / 1_000_000
if mb > 5:
    problems.append(f"The file is {mb:.1f} MB; over about 5 MB the page gets slow.")
print("Geometry types:", sorted(back.geom_type.unique()))
print("All checks passed" if not problems else "Not yet:\n- " + "\n- ".join(problems))

Geometry types: ['Point']
All checks passed


## 5) Preview the GeoJSON
Read the file back in and show the first feature.

In [8]:
with open(out_path, "r", encoding="utf-8") as f:
    data = json.load(f)
print("Features:", len(data.get("features", [])))
data["features"][0] if data.get("features") else {}

Features: 73


{'type': 'Feature',
 'geometry': {'type': 'Point', 'coordinates': [-73.580367, 45.520292]},
 'properties': {'photo': 'IMG_8621.HEIC',
  'image': 'photos/IMG_8621.jpg',
  'date': '2024-05-15 14:24',
  'lat': 45.520292,
  'lon': -73.580367,
  'name': '',
  'category': 'Cat',
  'description': ''}}

## 6) Next: open the map
1. In VS Code, right-click `site/index.html` and choose Open with Live Server.
2. `site/app.js` fetches `../data/cat_photos.geojson` and adds a pin for each photo. Click a pin to see the photo, its date and its coordinates.